# Outcome-map RV — the league

360 pre-declared configs in the real world and the same 360 in each placebo
world. Five expressions form a ladder, each rung stripping one component that
earlier work already showed is not a convergence trade:

| rung | what it trades |
|---|---|
| `pair_raw` | the raw cell richness — the control, which should collapse onto the already-dead short-dispersion trade |
| `pair_odd` | richness with the standing (even) premium projected out |
| `pair_odd_dev` | that, against the contract's own causal 20-session trailing tilt |
| `map_full` | the whole odd map as one telescoped book |
| `reswin` | `pair_odd` entered 1–5 sessions before a decision, exited after it |

crossed with the linear leg {none, ZQ basket, FOMC-swap package} — the axis
this whole study exists to price.

In [1]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../../")
sys.path.append(".")
from linvol_grid_common import pick_winner                  # noqa: E402
from RVUtils.SFRRVLab.stats import (                        # noqa: E402
    deflated_for_grid, grid_distribution, verdict)

DATA = Path("../data/outcome_map")
league = pd.read_parquet(DATA / "league.parquet")
real = league[league["world"] == "real"].reset_index(drop=True)
trades = pd.read_parquet(DATA / "trades_real.parquet")
with open(DATA / "dailies_real.pkl", "rb") as fh:
    dailies = pickle.load(fh)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
live = real[real["n_trades"] > 0]
print(f"configs: {len(real)} real ({len(live)} produced trades), "
      f"{len(league) - len(real)} placebo")
print(f"trades logged: {len(trades):,}")
print(f"trades per config: median {live['n_trades'].median():.0f}, "
      f"min {live['n_trades'].min():.0f}, max {live['n_trades'].max():.0f}")

configs: 360 real (360 produced trades), 720 placebo
trades logged: 17,916
trades per config: median 47, min 9, max 93


## 1. Does the expression run at all?

Channel-1 died on five trades in two years. The first thing to establish is
whether the cell instrument fixes that — because if it does not, nothing else
in this notebook matters.

In [2]:
print("=== trade count by expression and threshold ===")
print(real.pivot_table(index="expression", columns="thr_pp",
                       values="n_trades", aggfunc="median")
      .round(0).to_string())
print("\n=== the pre-declared kill criterion on trade count ===")
worst = live.groupby("expression")["n_trades"].median()
print(worst.round(0).to_string())
print(f"\nchannel-1 (PR #375) ran 5 trades in 2 years on the same signal "
      f"family. Median here: {live['n_trades'].median():.0f}.")

=== trade count by expression and threshold ===
thr_pp        4.0   8.0   16.0
expression                    
map_full      56.0  32.0  12.0
pair_odd      66.0  66.0  51.0
pair_odd_dev  66.0  62.0  46.0
pair_raw      65.0  63.0  56.0
reswin        32.0  32.0  22.0

=== the pre-declared kill criterion on trade count ===
expression
map_full        32.0
pair_odd        57.0
pair_odd_dev    56.0
pair_raw        57.0
reswin          28.0

channel-1 (PR #375) ran 5 trades in 2 years on the same signal family. Median here: 47.


## 2. The grid distribution — and the answer at 1x costs

In [3]:
dist = grid_distribution(live, metric="net_1x_bp")
print("=== net at 1x costs across the 360 configs ===")
for k, v in dist.items():
    print(f"  {k:14s} {v:,.2f}" if isinstance(v, float) else f"  {k:14s} {v}")
print("\n=== by expression (net_1x_bp) ===")
print(live.groupby("expression").agg(
    configs=("net_1x_bp", "size"), median=("net_1x_bp", "median"),
    best=("net_1x_bp", "max"), pct_pos=("net_1x_bp", lambda s: (s > 0).mean()),
    gross_med=("gross_bp", "median"),
    n_trades=("n_trades", "median")).round(2).to_string())

=== net at 1x costs across the 360 configs ===
  n_configs      360
  median         -132.46
  q25            -212.79
  q75            -79.26
  pct_positive   0.00
  best           -1.15
  worst          -425.87

=== by expression (net_1x_bp) ===
              configs  median   best  pct_pos  gross_med  n_trades
expression                                                        
map_full           72  -67.15  -3.19      0.0        0.0      32.0
pair_odd           72 -190.42  -1.15      0.0        0.0      57.0
pair_odd_dev       72 -166.08 -37.48      0.0        0.0      56.0
pair_raw           72 -185.95 -52.80      0.0        0.0      57.0
reswin             72  -91.03  -8.97      0.0        0.0      28.0


## 3. The linear leg, priced

The mission's question in one table. Every config exists three times — with no
linear leg, with the ZQ meeting basket, and with the FOMC-swap package — so
the leg's contribution and its bill are both matched-pair measurements, not
comparisons across different trades.

In [4]:
key = ["expression", "dte", "thr_pp", "exit", "direction"]
wide = live.pivot_table(index=key, columns="linear",
                        values=["gross_bp", "net_1x_bp", "lin_cost_bp",
                                "hedge_bp", "lin_contracts", "n_trades"])
print("=== the linear leg's contribution and bill (per config, medians) ===")
rows = []
for leg in ("zq", "swap"):
    if ("hedge_bp", leg) not in wide.columns:
        continue
    d_gross = (wide[("gross_bp", leg)] - wide[("gross_bp", "none")]).dropna()
    bill = wide[("lin_cost_bp", leg)].dropna()
    base_gross = wide[("gross_bp", "none")].dropna()
    rows.append({
        "leg": leg,
        "hedge_pnl_median_bp": round(float(wide[("hedge_bp", leg)].median()), 2),
        "gross_change_median_bp": round(float(d_gross.median()), 2),
        "bill_median_bp": round(float(bill.median()), 2),
        "bill_over_|gross|": round(float(
            (bill / base_gross.abs().replace(0, np.nan)).median()), 2),
        "contracts_median": round(float(
            wide[("lin_contracts", leg)].median()), 2),
        "configs_improved": round(float((d_gross > 0).mean()), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe 'bill' is the FedWatch convention of 3 ZQ legs per meeting.")
print("At 1 leg per meeting the bill is one third of the number above:")
print(live.groupby("linear")[["lin_cost_bp", "lin_cost_1leg_bp"]]
      .median().round(2).to_string())

=== the linear leg's contribution and bill (per config, medians) ===
 leg  hedge_pnl_median_bp  gross_change_median_bp  bill_median_bp  bill_over_|gross|  contracts_median  configs_improved
  zq                  0.0                     0.0           95.33               4.69              2.19               0.5
swap                  0.0                     0.0           94.51               4.84              2.19               0.5

The 'bill' is the FedWatch convention of 3 ZQ legs per meeting.
At 1 leg per meeting the bill is one third of the number above:
        lin_cost_bp  lin_cost_1leg_bp
linear                               
none           0.00              0.00
swap          94.51             31.50
zq            95.33             31.78


In [5]:
print("=== net at 1x by linear leg ===")
print(live.groupby("linear").agg(
    configs=("net_1x_bp", "size"), median_net=("net_1x_bp", "median"),
    best_net=("net_1x_bp", "max"), median_gross=("gross_bp", "median"),
    best_gross=("gross_bp", "max")).round(2).to_string())
print("\nThe all-config median gross is exactly zero because fade and momentum "
      "mirror: that IS the sign test passing, and it means the kill criterion "
      "has to be read on the fade half.")

=== net at 1x by linear leg ===
        configs  median_net  best_net  median_gross  best_gross
linear                                                         
none        120      -80.10     -3.19           0.0       64.99
swap        120     -185.73    -22.46           0.0       89.33
zq          120     -180.91     -1.15           0.0      143.76

The all-config median gross is exactly zero because fade and momentum mirror: that IS the sign test passing, and it means the kill criterion has to be read on the fade half.


In [6]:
fade = live[live["direction"] == "fade"]
med_gross_none = float(fade[fade["linear"] == "none"]["gross_bp"].median())
med_bill_zq = float(fade[fade["linear"] == "zq"]["lin_cost_bp"].median())
ratio = med_bill_zq / abs(med_gross_none) if med_gross_none else np.inf
print("PRE-DECLARED KILL CRITERION 1: the linear leg fails if its costs "
      "exceed half the gross of the paired expression.")
print(f"  median unhedged FADE gross {med_gross_none:+.1f}bp "
      f"vs median ZQ bill {med_bill_zq:.1f}bp -> ratio {ratio:.2f}x "
      f"({'FIRES' if med_bill_zq > 0.5 * abs(med_gross_none) else 'does not fire'})")
per = fade.copy()
per["per_trade_gross"] = per["gross_bp"] / per["n_trades"].replace(0, np.nan)
per["per_trade_optc"] = per["opt_cost_bp"] / per["n_trades"].replace(0, np.nan)
per["per_trade_linc"] = per["lin_cost_bp"] / per["n_trades"].replace(0, np.nan)
per["per_trade_hedge"] = per["hedge_bp"] / per["n_trades"].replace(0, np.nan)
per["per_trade_opt_gross"] = (per["opt_gross_bp"]
                              / per["n_trades"].replace(0, np.nan))
print("\n=== per-trade anatomy, fade only (bp per trade, medians) ===")
print(per.groupby(["expression", "linear"])[
    ["per_trade_opt_gross", "per_trade_hedge", "per_trade_gross",
     "per_trade_optc", "per_trade_linc"]].median().round(3).to_string())
print("\nThe option leg's P&L is identical across the three linear legs by "
      "construction (same signal, same trades, same marks), so every number in "
      "the hedge columns is a matched-pair measurement.")

PRE-DECLARED KILL CRITERION 1: the linear leg fails if its costs exceed half the gross of the paired expression.
  median unhedged FADE gross +13.6bp vs median ZQ bill 95.3bp -> ratio 7.02x (FIRES)

=== per-trade anatomy, fade only (bp per trade, medians) ===
                     per_trade_opt_gross  per_trade_hedge  per_trade_gross  per_trade_optc  per_trade_linc
expression   linear                                                                                       
map_full     none                  0.192            0.000            0.192           1.344           0.000
             swap                  0.192            0.116            0.357           1.344           1.766
             zq                    0.192            0.339            0.661           1.344           1.756
pair_odd     none                  0.307            0.000            0.307           1.801           0.000
             swap                  0.307            0.669            0.895           1.801        

## 4. The sign test

Both directions run on every config. A real edge shows as a mirror on gross:
whatever fade earns, momentum loses.

In [7]:
sign = live.pivot_table(index=["expression", "linear"], columns="direction",
                        values="gross_bp", aggfunc="median")
sign["mirror_error"] = (sign["fade"] + sign["momentum"]).abs()
print("=== median gross by direction (a mirror means the sign test passes) ===")
print(sign.round(2).to_string())

=== median gross by direction (a mirror means the sign test passes) ===
direction             fade  momentum  mirror_error
expression   linear                               
map_full     none    10.45    -10.45           0.0
             swap    15.45    -15.45           0.0
             zq      26.67    -26.67           0.0
pair_odd     none    22.59    -22.59           0.0
             swap    62.36    -62.36           0.0
             zq      69.71    -69.71           0.0
pair_odd_dev none    27.64    -27.64           0.0
             swap    34.35    -34.35           0.0
             zq      32.96    -32.96           0.0
pair_raw     none    27.12    -27.12           0.0
             swap    28.40    -28.40           0.0
             zq      39.69    -39.69           0.0
reswin       none   -11.78     11.78           0.0
             swap    16.78    -16.78           0.0
             zq      21.35    -21.35           0.0


## 5. The winner, deflated

`pick_winner` applies the n >= 10 floor before taking a maximum, so a
one-trade row cannot headline. DSR is computed at the FULL trial count.

In [8]:
w = pick_winner(live)
i = int(w.name)
d = dailies.get(i, pd.Series(dtype=float))
print("=== best n-floored config ===")
for k in ("expression", "dte", "thr_pp", "exit", "linear", "direction"):
    print(f"  {k:11s} {w[k]}")
print(f"  {'n_trades':11s} {int(w['n_trades'])}")
for k in ("gross_bp", "opt_gross_bp", "hedge_bp", "opt_cost_bp",
          "lin_cost_bp", "net_1x_bp", "net_2x_bp", "hit", "nw_t", "sharpe",
          "n_contracts", "hold_sessions", "converged"):
    print(f"  {k:11s} {w[k]:+.3f}" if np.isfinite(w[k]) else
          f"  {k:11s} n/a")

dsr = deflated_for_grid(d, real, sharpe_col="sharpe")
print(f"\nDSR at {dsr['n_trials']} trials: prob {dsr['dsr_prob']:.3f}, "
      f"annualised SR {dsr['sr_annualised']:.2f}")

med_cfg = float(live["net_1x_bp"].median())
v = verdict(net_bp_at_taker=float(w["net_1x_bp"]),
            net_bp_at_maker=float(w["gross_bp"]),
            dsr_prob=float(dsr["dsr_prob"]),
            median_net_bp=med_cfg, n_trades=int(w["n_trades"]))
print(f"median config {med_cfg:+.1f}bp -> HOUSE VERDICT: {v}")

=== best n-floored config ===
  expression  pair_odd
  dte         0-130
  thr_pp      16.0
  exit        hold
  linear      zq
  direction   fade
  n_trades    38
  gross_bp    +143.757
  opt_gross_bp +22.475
  hedge_bp    +121.282
  opt_cost_bp +64.500
  lin_cost_bp +80.405
  net_1x_bp   -1.148
  net_2x_bp   -146.053
  hit         +0.342
  nw_t        -0.018
  sharpe      -0.011
  n_contracts +6.789
  hold_sessions +16.000
  converged   +0.000

DSR at 360 trials: prob 0.000, annualised SR -0.01
median config -132.5bp -> HOUSE VERDICT: MARGINAL-maker-only


## 6. Verdict per expression

The same taxonomy applied to each rung of the ladder, so the question "did the
decomposition buy anything?" gets a per-rung answer rather than an impression.

In [9]:
rows = []
for expr, g in live.groupby("expression"):
    gw = pick_winner(g)
    gd = dailies.get(int(gw.name), pd.Series(dtype=float))
    gdsr = deflated_for_grid(gd, real, sharpe_col="sharpe")
    rows.append({
        "expression": expr, "configs": len(g), "n": int(gw["n_trades"]),
        "gross": round(float(gw["gross_bp"]), 1),
        "per_trade_gross": round(float(gw["gross_bp"] / max(gw["n_trades"], 1)),
                                 3),
        "net_1x": round(float(gw["net_1x_bp"]), 1),
        "net_2x": round(float(gw["net_2x_bp"]), 1),
        "median_cfg": round(float(g["net_1x_bp"].median()), 1),
        "nw_t": round(float(gw["nw_t"]), 2),
        "dsr": round(float(gdsr["dsr_prob"]), 3),
        "linear": gw["linear"],
        "verdict": verdict(net_bp_at_taker=float(gw["net_1x_bp"]),
                           net_bp_at_maker=float(gw["gross_bp"]),
                           dsr_prob=float(gdsr["dsr_prob"]),
                           median_net_bp=float(g["net_1x_bp"].median()),
                           n_trades=int(gw["n_trades"])),
    })
ladder = pd.DataFrame(rows).set_index("expression").loc[
    [e for e in ("pair_raw", "pair_odd", "pair_odd_dev", "map_full", "reswin")
     if e in [r["expression"] for r in rows]]]
print("=== the ladder, per rung ===")
print(ladder.to_string())
ladder.to_csv(DATA / "verdicts.csv")

=== the ladder, per rung ===
              configs   n  gross  per_trade_gross  net_1x  net_2x  median_cfg  nw_t  dsr linear              verdict
expression                                                                                                          
pair_raw           72  57   45.2            0.793   -52.8  -150.8      -185.9 -2.16  0.0   none  MARGINAL-maker-only
pair_odd           72  38  143.8            3.783    -1.1  -146.1      -190.4 -0.02  0.0     zq  MARGINAL-maker-only
pair_odd_dev       72  43   39.5            0.919   -37.5  -114.5      -166.1 -1.18  0.0   none  MARGINAL-maker-only
map_full           72  12   10.2            0.850    -3.2   -16.6       -67.1 -0.27  0.0   none  MARGINAL-maker-only
reswin             72  25   34.5            1.381    -9.0   -52.5       -91.0 -0.65  0.0   none  MARGINAL-maker-only


## 7. Cost curve — where, if anywhere, this clears

One backtest, the whole maker-to-taker curve: the gross is fixed and the
round-trip charge is swept, so the answer to "what execution would this need?"
is read rather than argued.

In [10]:
best_tr = trades[trades["config"] == i]
if len(best_tr):
    rt = float(best_tr["opt_cost_bp"].mean() + best_tr["lin_cost_bp"].mean())
    rows = []
    for mult in (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0):
        net = best_tr["gross_bp"] - mult * (best_tr["opt_cost_bp"]
                                            + best_tr["lin_cost_bp"])
        rows.append({"cost_mult": mult,
                     "round_trip_bp": round(mult * rt, 2),
                     "total_net_bp": round(float(net.sum()), 1),
                     "avg_net_bp": round(float(net.mean()), 3),
                     "hit": round(float((net > 0).mean()), 3)})
    print(f"=== cost curve for the winner ({len(best_tr)} trades, "
          f"1x round trip {rt:.2f}bp) ===")
    print(pd.DataFrame(rows).to_string(index=False))
    be = best_tr["gross_bp"].mean() / rt if rt > 0 else np.nan
    print(f"\nbreak-even cost multiple: {be:.2f}x "
          f"(1.0 = the house half-tick model)")

=== cost curve for the winner (38 trades, 1x round trip 3.81bp) ===
 cost_mult  round_trip_bp  total_net_bp  avg_net_bp   hit
      0.00           0.00         143.8       3.783 0.684
      0.25           0.95         107.5       2.830 0.500
      0.50           1.91          71.3       1.876 0.421
      0.75           2.86          35.1       0.923 0.368
      1.00           3.81          -1.1      -0.030 0.342
      1.50           5.72         -73.6      -1.937 0.289
      2.00           7.63        -146.1      -3.844 0.237

break-even cost multiple: 0.99x (1.0 = the house half-tick model)
